# Deploying and Operating Machine Learning Models
---
## Now You Try: Sample Solution

This notebook provides one valid solution to the learner exercise. Different candidate settings and deployment decisions may also be reasonable when they are supported by evidence.


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
from pathlib import Path

import atexit
import socket
import subprocess

import numpy as np
import pandas as pd
import requests
import sklearn
import mlflow
import mlflow.sklearn

from IPython.display import Markdown, display
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="mlflow")

print(f"Python: {sys.version.split()[0]}")
print(f"MLflow: {mlflow.__version__}")
print(f"scikit-learn: {sklearn.__version__}")

In [ ]:
def find_data_file(filename: str) -> Path:
    """Find the course data whether the notebook is run from the lesson folder or a notebooks folder."""
    candidates = [
        Path.cwd() / filename,
        Path.cwd() / "data" / filename,
        Path.cwd().parent / "data" / filename,
        Path("/mnt/data") / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(f"Could not find {filename}. Searched:\n{searched}")

DATA_PATH = find_data_file("SAheart.data")
WORK_DIR = (Path.cwd() / "mlops_artifacts").resolve()
MLRUNS_DIR = WORK_DIR / "mlruns"
LOG_DIR = WORK_DIR / "logs"

MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# A local file-based tracking store keeps this lesson self-contained.
TRACKING_URI = MLRUNS_DIR.as_uri()
mlflow.set_tracking_uri(TRACKING_URI)

print(f"Data: {DATA_PATH}")
print(f"Tracking store: {TRACKING_URI}")


In [ ]:
_BACKGROUND_PROCESSES = []


def port_is_open(port: int, host: str = "127.0.0.1") -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.25)
        return sock.connect_ex((host, port)) == 0


def wait_for_url(url: str, timeout: int = 30) -> bool:
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            response = requests.get(url, timeout=1)
            if response.status_code < 500:
                return True
        except requests.RequestException:
            pass
        time.sleep(0.5)
    return False


def start_background_process(command: list[str], log_name: str, env: dict | None = None):
    log_path = LOG_DIR / log_name
    log_file = open(log_path, "w", encoding="utf-8")
    process = subprocess.Popen(
        command,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        env=env,
        text=True,
    )
    _BACKGROUND_PROCESSES.append((process, log_file))
    return process, log_path


def cleanup_background_processes():
    for process, log_file in _BACKGROUND_PROCESSES:
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.kill()
        log_file.close()


atexit.register(cleanup_background_processes)

In [ ]:
heart = pd.read_csv(DATA_PATH, skipinitialspace=True)
heart["famhist"] = (heart["famhist"] == "Present").astype(int)

X = heart.drop(columns=["row.names", "chd"])
y = heart["chd"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=2022,
    stratify=y,
)

print(f"Rows: {len(heart):,}")
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")
X_train.head()

# Part 1: Complete the run-logging function


In [ ]:
EXPERIMENT_NAME = "heart-disease-mlops-exercise"
mlflow.set_experiment(EXPERIMENT_NAME)


def fit_and_log_run(
    *,
    run_name: str,
    C: float,
    penalty: str,
    class_weight=None,
) -> dict:
    model = Pipeline([
        ("sc", StandardScaler()),
        (
            "lr",
            LogisticRegression(
                C=C,
                penalty=penalty,
                solver="liblinear",
                class_weight=class_weight,
                max_iter=2_000,
                random_state=2022,
            ),
        ),
    ])

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, predictions)
    roc_auc = roc_auc_score(y_test, probabilities)

    input_example = X_test.iloc[:3].copy()
    signature = infer_signature(input_example, model.predict(input_example))

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_params({
            "model_type": "LogisticRegression",
            "scaler": "StandardScaler",
            "C": C,
            "penalty": penalty,
            "solver": "liblinear",
            "class_weight": str(class_weight),
            "random_state": 2022,
        })
        mlflow.log_metrics({
            "accuracy": accuracy,
            "roc_auc": roc_auc,
        })
        mlflow.set_tags({
            "exercise": "compare-and-deploy",
            "owner": "data-science-learner",
            "deployment_pattern": "local-online-endpoint",
        })

        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name="heart_disease_pipeline",
            signature=signature,
            input_example=input_example,
        )

        run_id = run.info.run_id
        model_uri = model_info.model_uri

    return {
        "run_name": run_name,
        "run_id": run_id,
        "model_uri": model_uri,
        "accuracy": accuracy,
        "roc_auc": roc_auc,
        "C": C,
        "penalty": penalty,
        "class_weight": class_weight,
    }

# Part 2: Log two runs


In [ ]:
baseline_result = fit_and_log_run(
    run_name="baseline",
    C=0.123,
    penalty="l1",
    class_weight=None,
)

candidate_result = fit_and_log_run(
    run_name="candidate-balanced-l2",
    C=1.0,
    penalty="l2",
    class_weight="balanced",
)

results = pd.DataFrame([baseline_result, candidate_result])
results[["run_name", "accuracy", "roc_auc", "C", "penalty", "class_weight"]]

# Part 3: Select a model


In [ ]:
selected_result = max(
    [baseline_result, candidate_result],
    key=lambda result: result["roc_auc"],
)

model_uri = selected_result["model_uri"]
print(f"Selected run: {selected_result['run_name']}")
print(f"ROC AUC: {selected_result['roc_auc']:.3f}")
print(f"Model URI: {model_uri}")

### Sample deployment decision

Select the run with the higher test ROC AUC for this classroom exercise. Before a real release, also examine confidence intervals, subgroup performance, calibration, operating thresholds, business costs of errors, and evidence from a larger validation sample.

# Part 4: Inspect the experiment


In [ ]:
UI_PORT = 5000
UI_URL = f"http://127.0.0.1:{UI_PORT}"

venv_bin = str(Path(sys.executable).parent)

env = os.environ.copy()
env["MLFLOW_TRACKING_URI"] = TRACKING_URI
env["PATH"] = venv_bin + os.pathsep + env.get("PATH", "")

if port_is_open(UI_PORT):
    print(f"Port {UI_PORT} is already in use. Open {UI_URL} and check whether MLflow is running there.")
else:
    ui_command = [
        sys.executable,
        "-m",
        "mlflow",
        "ui",
        "--backend-store-uri",
        TRACKING_URI,
        "--host",
        "127.0.0.1",
        "--port",
        str(UI_PORT),
    ]
    ui_process, ui_log = start_background_process(ui_command, "mlflow-ui.log", env=env)
    if wait_for_url(UI_URL, timeout=30):
        print(f"MLflow UI is running at {UI_URL}")
    else:
        print(f"MLflow UI did not start. Inspect the log: {ui_log}")

# Part 5: Serve and test the selected model


In [ ]:
MODEL_PORT = 5001
MODEL_BASE_URL = f"http://127.0.0.1:{MODEL_PORT}"

if port_is_open(MODEL_PORT):
    raise RuntimeError(
        f"Port {MODEL_PORT} is already in use. Stop the existing process or change MODEL_PORT."
    )

venv_bin = str(Path(sys.executable).parent)

env = os.environ.copy()
env["MLFLOW_TRACKING_URI"] = TRACKING_URI
env["PATH"] = venv_bin + os.pathsep + env.get("PATH", "")

serve_command = [
    sys.executable,
    "-m",
    "mlflow",
    "models",
    "serve",
    "--model-uri",
    model_uri,
    "--host",
    "127.0.0.1",
    "--port",
    str(MODEL_PORT),
    "--env-manager",
    "local",
]

serve_process, serve_log = start_background_process(
    serve_command,
    "mlflow-model-server.log",
    env=env,
)

if wait_for_url(f"{MODEL_BASE_URL}/health", timeout=60):
    print(f"Model server is healthy at {MODEL_BASE_URL}")
else:
    raise RuntimeError(f"Model server did not start. Inspect the log: {serve_log}")

In [ ]:
request_rows = X_test.iloc[:2].copy()
payload = {
    "dataframe_split": request_rows.to_dict(orient="split")
}

response = requests.post(
    f"{MODEL_BASE_URL}/invocations",
    json=payload,
    timeout=10,
)

print(f"Status code: {response.status_code}")
print(json.dumps(response.json(), indent=2))

# Optional stretch: test an invalid request


In [ ]:
invalid_rows = request_rows.rename(columns={"age": "customer_age"})
invalid_payload = {
    "dataframe_split": invalid_rows.to_dict(orient="split")
}

invalid_response = requests.post(
    f"{MODEL_BASE_URL}/invocations",
    json=invalid_payload,
    timeout=10,
)

print(f"Status code: {invalid_response.status_code}")
print(invalid_response.text[:1_500])

# Sample monitoring plan

| Question | Sample answer |
|---|---|
| Service-health metric | Availability, request error rate, and response time |
| Data-health signal | Missing fields and changes in distributions of age, tobacco, and LDL |
| Model/business outcome | ROC AUC when outcomes arrive; recall at the agreed intervention threshold |
| Alert trigger | Error rate or latency exceeds the agreed service objective; drift exceeds its threshold |
| Rollback/retraining trigger | New release fails validation, or sustained performance decline is confirmed |
| Owner | Named model owner supported by engineering and the business process owner |


## Cleanup


In [ ]:
cleanup_background_processes()
print("Background MLflow processes stopped.")